# 02 — Event Study
Abnormal returns, CAR, t-tests, asymmetry tests around 10 oil-shock events.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from event_study import (
    OIL_SHOCK_EVENTS, run_event_study,
    t_test_car, asymmetry_test, oil_sensitivity,
    get_sector_returns, winsorize_returns,
)
from utils import (
    DATA_PROC, TABLES_DIR, PLOTS_DIR, set_theme, save_table,
    plot_avg_car, plot_car_heatmap,
)
from constants import MARKET_COL, OIL_COL, FX_COL, SECTORS

print('Events in catalogue:', len(OIL_SHOCK_EVENTS))
print('Sectors:', SECTORS)


Events in catalogue: 10
Sectors: ['OIL_GAS', 'AUTO', 'FMCG', 'IT', 'PHARMA']


In [2]:
# Load stock-level returns; get_sector_returns aggregates to equal-weight sectors
returns_raw = pd.read_parquet(DATA_PROC / 'returns.parquet')
print(f'Stock returns : {returns_raw.shape}  cols: {list(returns_raw.columns)}')

returns = get_sector_returns(returns_raw)
print(f'Sector returns: {returns.shape}  cols: {list(returns.columns)}')


Stock returns : (1928, 18)  cols: ['RELIANCE', 'ONGC', 'IOC', 'BPCL', 'INDIGO', 'HPCL', 'ADANIPORTS', 'TATAMOTORS', 'MARUTI', 'ASIANPAINT', 'HINDUNILVR', 'ITC', 'TCS', 'INFY', 'CIPLA', 'BRENT', 'NIFTY', 'USDINR']
Sector returns: (1928, 8)  cols: ['OIL_GAS', 'AUTO', 'FMCG', 'IT', 'PHARMA', 'NIFTY', 'BRENT', 'USDINR']


In [3]:
# Run event study — run_event_study accepts raw stock-level returns and
# aggregates internally.  winsorize=True clips BRENT outliers at ±25%.
results = run_event_study(
    returns_raw,
    OIL_SHOCK_EVENTS,
    pre=5, post=5,
    market_col=MARKET_COL,
    winsorize=True,
)
print(f'Events processed: {len(results["ar_all"])}')
print('\nAverage CAR across all events (%):\n', (results['avg_car'] * 100).round(3).to_string())


Events processed: 10

Average CAR across all events (%):
 OIL_GAS    0.137
AUTO      -0.132
FMCG       0.157
IT        -0.220
PHARMA    -2.015


In [4]:
# Average cumulative AR plot
plot_avg_car(results['avg_ar'])


Saved → C:\Users\anves\projects\osi_clean\outputs\plots\avg_car.png


In [5]:
# CAR heatmap: event × sector
plot_car_heatmap(results['car_all'])


Saved → C:\Users\anves\projects\osi_clean\outputs\plots\car_heatmap.png


In [6]:
# Statistical significance — one-sample t-test H0: CAR = 0
sig_rows = [t_test_car(results['car_all'], s) for s in SECTORS]
sig_df   = pd.DataFrame(sig_rows)
save_table(sig_df, 'significance_test')
sig_df[['sector','n_events','mean_car_pct','t_stat','p_value','significant']]


Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\significance_test.csv


,sector,n_events,mean_car_pct,t_stat,p_value,significant
0,OIL_GAS,10,0.137,0.093,0.9278,False
1,AUTO,10,-0.132,-0.069,0.9467,False
2,FMCG,10,0.157,0.109,0.9156,False
3,IT,10,-0.220,-0.137,0.8940,False
4,PHARMA,10,-2.015,-0.985,0.3504,False


In [7]:
# Asymmetry test: up-shock CAR vs down-shock CAR (Welch's t-test)
asym_rows = [asymmetry_test(results['car_all'], s) for s in SECTORS]
asym_df   = pd.DataFrame(asym_rows)
save_table(asym_df, 'asymmetry_test')
asym_df


Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\asymmetry_test.csv


,sector,n_up,n_down,mean_car_up,mean_car_up_pct,mean_car_down,mean_car_down_pct,t_stat,p_value,asymmetric
0,OIL_GAS,7,3,-0.0105,-1.048,0.0290,2.902,-1.173,0.3191,False
1,AUTO,7,3,-0.0079,-0.793,0.0141,1.412,-0.344,0.7605,False
2,FMCG,7,3,-0.0205,-2.048,0.0530,5.300,-2.901,0.0703,True
3,IT,7,3,-0.0021,-0.213,-0.0024,-0.237,0.004,0.9968,False
4,PHARMA,7,3,-0.0076,-0.760,-0.0494,-4.944,0.745,0.5172,False


In [8]:
# Day-by-day average AR table
avg_ar_pct = (results['avg_ar'] * 100).round(3)
save_table(avg_ar_pct.reset_index(), 'avg_ar_by_day')
print('NB02 complete ✓')
avg_ar_pct


Saved table → C:\Users\anves\projects\osi_clean\outputs\tables\avg_ar_by_day.csv
NB02 complete ✓


,OIL_GAS,AUTO,FMCG,IT,PHARMA
-5,-0.582,-0.419,-0.272,-0.289,-1.030
-4,0.777,0.255,1.084,0.357,0.884
-3,-0.276,-0.381,-0.732,-0.486,-0.387
-2,0.116,0.266,0.092,0.246,-0.518
-1,0.327,0.539,0.239,0.058,-0.285
0,0.013,-0.344,-0.166,0.225,0.034
1,-0.596,-0.548,-0.484,-0.233,-1.254
2,0.020,-0.340,-0.091,0.065,1.086
3,-0.316,-0.224,-0.348,-0.019,-0.597
4,0.555,0.592,0.431,0.295,0.733
